For the EEG Experiment the design matrix needs to:
- contain double the amount of neutral cues compared to the other 2
- use only 4 images, either (_00 or _01)



For the EEG Experiment we need to come up with a scheme for the triggers 
We will have a 2 x 3 x 4 design (Masking x Expectation x images) - this gives me 24 unique triggers 
The triggers with Brain products can vary from 1 x 250 


- images: 0-1-2-3 \\
- mask: + 10-20 \\
- expectation: decimals, 100s, 200s (neutral, expected, unexpected)\\


11 -> image 1: early-mask, neutral \
111-> image 1: early-mask, expected \
211-> image 1: early-mask, unexpected \

21 -> image 1: late-mask, neutral \
121-> image 1: late-mask, expected \
221-> image 1: late-mask, unexpected \


In [11]:
import numpy as np
import os
import random
import pandas as pd 
from collections import Counter

def create_cue_dynam(highProb=0.7, lowProb=0.3, neutral=1.0, trials_per_cue=40):
    
    # double the amount for EEG 
    trial_per_neutral = 2 * trials_per_cue
    
    cue_data = {"cue_names": [".\\cues\\Sea_Animal.png",  ".\\cues\\Water_Vehicle.png",  ".\\cues\\Neutral.png"],
                "cue_highProb_cats": [["dolphin", "whale"], ["speedboat", "submarine"], 
                                    ["dolphin", "whale", "speedboat", "submarine"]],
                
                "cue_lowProb_cats": [["speedboat", "submarine"], ["dolphin", "whale"],
                                    ["dolphin", "whale", "speedboat", "submarine"]]}

    cue_data["cue_highProb"] = []
    cue_data["cue_lowProb"] = []
    
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        if cue != ".\\cues\\Neutral.png":
            cue_data["cue_highProb"].append([np.round(highProb / len(cue_data["cue_highProb_cats"][cue_id]), 2)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(lowProb / len(cue_data["cue_lowProb_cats"][cue_id]), 2)] * len(cue_data["cue_lowProb_cats"][cue_id]))
        
        else:
            cue_data["cue_highProb"].append([np.round(neutral / len(cue_data["cue_highProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(neutral / len(cue_data["cue_lowProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            
    cue_data["high_prob_trials"] = [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_highProb"])]
    
    cue_data["low_prob_trials"] =  [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_lowProb"])]
    
    return cue_data

def assign_trigger(row, late=0.100):
    
    # ---- MASK ----
    if row['mask_ISI'] == 0.017:
        mask_code = 10
    elif row['mask_ISI'] == late:
        mask_code = 20
    else:
        raise ValueError("Unknown mask type")
    
    # ---- IMAGE (side dependent) ----
    image_code = row['image_index']
    base_code = mask_code + image_code
    
    # ---- EXPECTATION ----
    if row['expectation'] == 'neutral':
        exp_code = 0
    elif row['expectation'] == 'expected':
        exp_code = 100
    elif row['expectation'] == 'unexpected':
        exp_code = 200
    else:
        raise ValueError("Unknown expectation")
    
    return base_code + exp_code
def allocate_catch_trials(N, p=0.3, k=3, alpha=0.66):
    C = int(N * p)
    
    # ensure divisible by (k-1)*2 if needed
    while C % (k-1) != 0:
        C -= 1
    
    main = int(round(alpha * C))
    
    # enforce even split
    remainder = C - main
    other = remainder // (k - 1)
    
    # adjust if rounding broke divisibility
    main = C - other * (k - 1)
    other = [other] * (k - 1)
    
    return main, other[0], other[1]

def assign_catch_trials(df, rng=None):
    """
    Takes your existing 320-trial dataframe and:
      1. Flags 80 rows as catch trials (52 neutral, 14 expected, 14 unexpected),
         balanced across mask_ISI within each expectation condition.
      2. Reorders the sequence so catches are spaced min=2, max=7-8, mean~4 apart.
    
    Adds an 'identity_catch' boolean column.
    Returns a reordered dataframe (reset index).
    """
    if rng is None:
        rng = random.Random()

    df = df.copy()
    df['identity_catch'] = False

    # --- 1. Flag catch trials ---
    neut, exp, unexp = allocate_catch_trials(len(df), 0.25, 3, 0.64)
    catch_counts = {'neutral': neut, 'expected': exp, 'unexpected': unexp}

    for condition, n_catches in catch_counts.items():
        cond_idx = df[df['expectation'] == condition].index.tolist()
        assert len(cond_idx) >= n_catches, \
            f"Not enough {condition} trials: need {n_catches}, have {len(cond_idx)}"

        # Balance across mask_ISI: half from 0.017, half from 0.100
        half = n_catches // 2  # both 52 and 14 are even, so no remainder

        for isi_val, n in [(0.017, half), (0.100, half)]:
            isi_idx = df.loc[cond_idx][df.loc[cond_idx, 'mask_ISI'] == isi_val].index.tolist()
            assert len(isi_idx) >= n, \
                f"Not enough {condition}/ISI={isi_val} trials: need {n}, have {len(isi_idx)}"
            chosen = rng.sample(isi_idx, n)
            df.loc[chosen, 'identity_catch'] = True

    # --- 2. Separate catches and non-catches ---
    catch_df    = df[df['identity_catch']].sample(frac=1, random_state=rng.randint(0, 99999)).reset_index(drop=True)
    noncatch_df = df[~df['identity_catch']].sample(frac=1, random_state=rng.randint(0, 99999)).reset_index(drop=True)

    n_catches  = len(catch_df)    # 80
    n_noncatch = len(noncatch_df) # 240

    # --- 3. Sample inter-catch gaps ---
    # gap = number of non-catch trials BEFORE each catch
    # gap in [1, 7] → total catch-to-catch distance of [2, 8], mean ~4
    gaps = _sample_gaps(
        n_catches  = n_catches,
        n_noncatch = n_noncatch,
        gap_min    = 1,
        gap_max    = 7,
        gap_mean   = 3.0,  # 3 non-catches between → distance of 4
        rng        = rng,
    )

    # --- 4. Interleave into final sequence ---
    sequence_rows = []
    nc_pointer = 0

    for i, gap in enumerate(gaps):
        # Insert `gap` non-catch trials
        for _ in range(gap):
            sequence_rows.append(noncatch_df.iloc[nc_pointer])
            nc_pointer += 1
        # Insert catch trial
        sequence_rows.append(catch_df.iloc[i])

    # Append any leftover non-catch trials at the end
    while nc_pointer < n_noncatch:
        sequence_rows.append(noncatch_df.iloc[nc_pointer])
        nc_pointer += 1

    result = pd.DataFrame(sequence_rows).reset_index(drop=True)
    return result


def _sample_gaps(n_catches, n_noncatch, gap_min, gap_max, gap_mean, rng):
    """
    Returns a list of n_catches integers in [gap_min, gap_max]
    that sum exactly to n_noncatch, distributed around gap_mean.
    """
    # Sanity check: is the target sum achievable?
    assert gap_min * n_catches <= n_noncatch <= gap_max * n_catches, (
        f"Cannot distribute {n_noncatch} non-catch trials across {n_catches} gaps "
        f"with min={gap_min}, max={gap_max}. "
        f"Feasible range: [{gap_min * n_catches}, {gap_max * n_catches}]"
    )

    for _ in range(50_000):
        gaps = [
            max(gap_min, min(gap_max, round(rng.triangular(gap_min, gap_max, gap_mean))))
            for _ in range(n_catches)
        ]
        diff = sum(gaps) - n_noncatch

        # Nudge gaps up or down to hit the exact sum
        for _ in range(5_000):
            if diff == 0:
                break
            idx = rng.randrange(n_catches)
            if diff > 0 and gaps[idx] > gap_min:
                gaps[idx] -= 1
                diff -= 1
            elif diff < 0 and gaps[idx] < gap_max:
                gaps[idx] += 1
                diff += 1

        if diff == 0:
            return gaps

    raise RuntimeError("Failed to converge on valid gap distribution.")

def validate_sequence(df):
    catches = df[df['identity_catch']]
    print(f"Total:     {len(df)}")
    print(f"Catch:     {len(catches)}  ({len(catches)/len(df)*100:.1f}%)")
    print(f"Non-catch: {len(df) - len(catches)}")

    print(f"\nCatch breakdown by expectation:")
    for cond, grp in catches.groupby('expectation'):
        c017 = (grp['mask_ISI'] == 0.017).sum()
        c100 = (grp['mask_ISI'] == 0.100).sum()
        print(f"  {cond:11s}: total={len(grp)}  ISI=0.017: {c017}  ISI=0.100: {c100}")

    catch_pos = df.index[df['identity_catch']].tolist()
    distances = [catch_pos[i+1] - catch_pos[i] for i in range(len(catch_pos) - 1)]
    print(f"\nCatch-to-catch distances:")
    print(f"  min={min(distances)}  max={max(distances)}  mean={np.mean(distances):.2f}")
    print(f"  distribution: {dict(sorted(Counter(distances).items()))}")

def build_constrained_order(df, rng=None, max_unexpected_run=1):
    
    if rng is None:
        rng = random.Random()

    remaining = df.copy()
    ordered_rows = []

    last_target = None
    unexpected_run = 0

    while len(remaining) > 0:

        # valid candidates mask
        valid_mask = np.ones(len(remaining), dtype=bool)

        # Rule 1 — no same target twice
        if last_target is not None:
            valid_mask &= (remaining["target"].values != last_target)

        # Rule 2 — max unexpected run
        if unexpected_run >= max_unexpected_run:
            valid_mask &= (remaining["expectation"].values != "unexpected")

        valid = remaining[valid_mask]

        # if dead end → restart whole sequence
        if len(valid) == 0:
            return build_constrained_order(
                df,
                rng=random.Random(rng.randrange(int(1e9))),
                max_unexpected_run=max_unexpected_run
            )

        # pick random valid row
        choice_idx = rng.randrange(len(valid))
        row = valid.iloc[choice_idx]

        ordered_rows.append(row)

        # update state
        last_target = row["target"]
        if row["expectation"] == "unexpected":
            unexpected_run += 1
        else:
            unexpected_run = 0

        # remove selected row
        remaining = remaining.drop(valid.index[choice_idx])

    return pd.DataFrame(ordered_rows).reset_index(drop=True)

def create_block_trials(stim_path, cue_data, random_seed, long_isi=0.1, pick_images="_01", identity_catch=0.1): 
    
    rng = random.Random(random_seed) 
    categories = os.listdir(stim_path)
    stimuli = []
    for cat in categories:
        cat_path = os.path.join(stim_path, cat)
        files = os.listdir(cat_path)
        stimuli.extend([f".\\stimuli\\{cat}\\{x}" for x in files])

    # Filter stims
    stimuli = np.array(stimuli)
    stimuli = np.array([x for x in stimuli if pick_images in x ])
    stimuli = stimuli[np.argsort(stimuli)]

    data = {"target_id": [],
            "target": [],
            "expectation": [],
            "mask_ISI": [],
            "cue": [],
            "target_name": [],
            "target_cat": []}

    mask_type = [0.017, long_isi]
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        high_cats = np.array(cue_data["cue_highProb_cats"][cue_id])
        low_cats = np.array(cue_data["cue_lowProb_cats"][cue_id])

        # since the neutral category has all 4 images there is no need to repeat it twice
        if cue != ".\\cues\\Neutral.png":
            # This loop handles only unexpexted cases
            for i, l_cat in enumerate(low_cats):
                l_cat_stim = stimuli[np.char.count(stimuli, l_cat) > 0]
                targets = l_cat_stim[np.char.count(l_cat_stim, "mask") == 0]
                target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]
    
                for mask in mask_type:
                
                    h_trials = int(cue_data["low_prob_trials"][cue_id][i])
                
                    data["target_id"].extend(np.repeat(target_ids , h_trials))
                    data["target"].extend(np.repeat(targets, h_trials))
                    
                    target_names =[x.split("\\")[-1] for x in targets]
                    target_categories = [x.split("_")[0] for x in target_names]               

                    data["target_name"].extend(np.repeat(target_names , h_trials))
                    data["target_cat"].extend(np.repeat(target_categories , h_trials))
                    data["expectation"].extend(["unexpected"] * h_trials)       
                    data["mask_ISI"].extend([mask] * h_trials)
                    data["cue"].extend([cue] * h_trials)
        
        # This loop handles expexted and neutral cases         
        for i, h_cat in enumerate(high_cats):
            h_cat_stim = stimuli[np.char.count(stimuli, h_cat) > 0]
            targets = h_cat_stim[np.char.count(h_cat_stim, "mask") == 0]
            target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

            for mask in mask_type:
                h_trials = int(cue_data["high_prob_trials"][cue_id][i])
                
                data["target_id"].extend(np.repeat(target_ids , h_trials))
                data["target"].extend(np.repeat(targets , h_trials))
                
                target_names =[x.split("\\")[-1] for x in targets]
                target_categories = [x.split("_")[0] for x in target_names]
            
                data["target_name"].extend(np.repeat(target_names , h_trials))
                data["target_cat"].extend(np.repeat(target_categories , h_trials))
                
                if cue != ".\\cues\\Neutral.png":
                    data["expectation"].extend(["expected"] * h_trials)
                else:
                    data["expectation"].extend(["neutral"]* h_trials)
                    
                data["mask_ISI"].extend([mask] * h_trials)
                data["cue"].extend([cue] * h_trials)
            
    mapping = {0: 1,
               3: 2,
               1: 3,
               2: 4}
    
    df = pd.DataFrame(data)
    df['image_index'] = df['target_id'].map(mapping)
    df['trigger'] = df.apply(assign_trigger, axis=1)
    df = build_constrained_order(df, rng=rng)
    df = assign_catch_trials(df, rng=rng)
    validate_sequence(df)
    
    return df, stimuli

In [17]:
cue_data = create_cue_dynam()
stim_path = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_EEG_Task/stimuli"
trials, stimss = create_block_trials(stim_path, cue_data, random_seed=23)

Total:     320
Catch:     80  (25.0%)
Non-catch: 240

Catch breakdown by expectation:
  expected   : total=14  ISI=0.017: 7  ISI=0.100: 7
  neutral    : total=52  ISI=0.017: 26  ISI=0.100: 26
  unexpected : total=14  ISI=0.017: 7  ISI=0.100: 7

Catch-to-catch distances:
  min=2  max=7  mean=3.99
  distribution: {2: 15, 3: 19, 4: 15, 5: 14, 6: 14, 7: 2}


In [18]:
trials.head(10)

,target_id,target,expectation,mask_ISI,cue,target_name,target_cat,image_index,trigger,identity_catch
0,3,.\stimuli\whale\whale_01.jpg,expected,0.100,.\cues\Sea_Animal.png,whale_01.jpg,whale,2,122,False
1,0,.\stimuli\dolphin\dolphin_01.jpg,expected,0.017,.\cues\Sea_Animal.png,dolphin_01.jpg,dolphin,1,111,False
2,1,.\stimuli\speedboat\speedboat_01.jpg,neutral,0.100,.\cues\Neutral.png,speedboat_01.jpg,speedboat,3,23,False
3,2,.\stimuli\submarine\submarine_01.jpg,neutral,0.017,.\cues\Neutral.png,submarine_01.jpg,submarine,4,14,False
4,3,.\stimuli\whale\whale_01.jpg,unexpected,0.017,.\cues\Water_Vehicle.png,whale_01.jpg,whale,2,212,True
5,0,.\stimuli\dolphin\dolphin_01.jpg,neutral,0.100,.\cues\Neutral.png,dolphin_01.jpg,dolphin,1,21,False
6,1,.\stimuli\speedboat\speedboat_01.jpg,neutral,0.017,.\cues\Neutral.png,speedboat_01.jpg,speedboat,3,13,False
7,2,.\stimuli\submarine\submarine_01.jpg,expected,0.100,.\cues\Water_Vehicle.png,submarine_01.jpg,submarine,4,124,False
8,0,.\stimuli\dolphin\dolphin_01.jpg,neutral,0.100,.\cues\Neutral.png,dolphin_01.jpg,dolphin,1,21,True
9,0,.\stimuli\dolphin\dolphin_01.jpg,expected,0.100,.\cues\Sea_Animal.png,dolphin_01.jpg,dolphin,1,121,False


In [12]:
unique_combos = trials[['trigger', 'target_name', 'target_id']].drop_duplicates()
unique_combos.sort_values("trigger")

,trigger,target_name,target_id
15,11,dolphin_01.jpg,0
18,12,whale_01.jpg,3
11,13,speedboat_01.jpg,1
6,14,submarine_01.jpg,2
0,21,dolphin_01.jpg,0
13,22,whale_01.jpg,3
32,23,speedboat_01.jpg,1
3,24,submarine_01.jpg,2
7,111,dolphin_01.jpg,0
10,112,whale_01.jpg,3


In [1]:
import os
from PIL import Image

root_dir = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/target_stimuli"
target_size = (500, 500)

# Image extensions to process
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower().endswith(valid_exts):
            img_path = os.path.join(root, file)

            try:
                with Image.open(img_path) as img:
                    img = img.convert("RGB")  # safe for consistency
                    img_resized = img.resize(target_size, Image.LANCZOS)
                    img_resized.save(img_path)

            except Exception as e:
                print(f"Failed to process {img_path}: {e}")
